In [1]:
import pyroomacoustics as pra

import os
from tqdm import tqdm

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.autograd import profiler
import torchaudio
from torchmetrics.audio import SpeechReverberationModulationEnergyRatio, ShortTimeObjectiveIntelligibility, DeepNoiseSuppressionMeanOpinionScore, ScaleInvariantSignalDistortionRatio


from einops import rearrange

from src.dataset import SignalDataset, TRUNetDataset
from src.loss import loss_tot, loss_MR, loss_MR_w
from models.fspen import * # FullSubPathExtension, FullSubPathExtension_3_heads, FullSubPathExtension_ver2, FullSubPathExtension_abs_pha, FullSubPathExtension_abs_pha_mapping, FullSubPathExtension_ver2_abs_pha, FullSubPathExtension_ver3

from IPython.display import Audio

from src.utils import model_eval, model_eval_fspen2x_ver3, model_eval_3_heads, use_pcs, inv_pcs, model_eval_old

import matplotlib.pyplot as plt

torch.set_num_threads(1)
torch.set_num_interop_threads(1)
torch._logging.set_logs(graph_code=False)

In [2]:
TEST_DIR = os.path.join("data", "DS_10283_2791", "clean_testset_wav")
TEST_NOISE_DIR = os.path.join("data", "DS_10283_2791", "noisy_testset_wav")
NOISE_DIR = os.path.join("data", "demand_test")

CHKP_DIR = "checkpoints"

np.set_printoptions(precision=3)
torch.set_printoptions(precision=3)

In [3]:
import random

SEED = 1984

np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

In [4]:
from src.fspen_configs import *

configs = TrainConfig_48kHz_overlap() # TrainConfig_explicit_unfold()
# print(sum(configs.bands_num_in_groups), configs.dual_path_extension["num_modules"])
fspen = FullSubPathExtension(configs=configs)# .to(DEVICE)

state_d = torch.load(os.path.join(CHKP_DIR, "fspen_chkp", "TrainConfig_48kHz_overlap_1986#0.pt"), map_location="cpu",  weights_only=False)
fspen.eval()

FullSubPathExtension(
  (full_band_encoder): FullBandEncoder(
    (full_band_encoder): ModuleList(
      (0): FullBandEncoderBlock(
        (conv): Conv1d(2, 4, kernel_size=(6,), stride=(2,), padding=(2,))
        (norm): BatchNorm1d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (activate): ELU(alpha=1.0)
      )
      (1): FullBandEncoderBlock(
        (conv): Conv1d(4, 16, kernel_size=(8,), stride=(2,), padding=(3,))
        (norm): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (activate): ELU(alpha=1.0)
      )
      (2): FullBandEncoderBlock(
        (conv): Conv1d(16, 32, kernel_size=(6,), stride=(2,), padding=(2,))
        (norm): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (activate): ELU(alpha=1.0)
      )
    )
    (global_features): Conv1d(32, 32, kernel_size=(1,), stride=(1,))
  )
  (sub_band_encoder): SubBandEncoder_baseline(
    (sub_band_encoders): ModuleList

In [5]:
fspen.load_state_dict(state_d["model_state_dict"])

<All keys matched successfully>

In [6]:
# fspen = torch.compile(fspen, mode="reduce-overhead")

h0 = [[torch.zeros(configs.dual_path_extension["parameters"]["num_layers"], 1 * configs.num_bands_out, configs.dual_path_extension["parameters"]["inter_hidden_size"]) for _ in range(8)] for _ in range(configs.dual_path_extension["num_modules"])]


input_spec_ = torch.ones(1, 1, 2, 513)
abs_spectrum = torch.ones(1, 1, 1, 513)

In [7]:
# N_FFTS = 512
# HOP_LENGTH = 256
# HID_SIZE = 32
# SR = 16_000

N_FFTS = configs.n_fft
HOP_LENGTH = configs.hop_length
HID_SIZE = 64
SR = configs.sample_rate
BATCH_SIZE = 8 # 32

DEVICE = "cpu" # torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"It's {DEVICE} time!!!")

It's cpu time!!!


In [8]:
rir_dict = {1: os.path.join("data", "rirs48_small_3_test"), 1: os.path.join("data", "rirs48_medium_3_test"), 1: os.path.join("data", "rirs48_large_3_test"), 1: os.path.join("data", "rirs48_super_large_3_test")}
dataset = TRUNetDataset(TEST_DIR, sr=SR, noise_dir=NOISE_DIR, rir_dir=rir_dict, snr=[0, 5, 10, 15], rir_proba=0.85, noise_proba=0.85, rir_target=False, return_noise=False, return_rir=False, verbose=False)
dataset.set_epoch(99)

180
12


In [9]:
def vorbis_window(winlen, device="cuda"):
    sq = torch.sin(torch.pi/2*(torch.sin(torch.pi/winlen*(torch.arange(winlen)-0.5))**2)).float()
    return sq

In [10]:
import yaml

from NISQA_s.src.core.model_torch import model_init
from NISQA_s.src.utils.process_utils import process

NISQA_PATH = "NISQA_s/config/nisqa_s.yaml"

with open(NISQA_PATH, 'r') as stream:
    nisqa_args = yaml.safe_load(stream)
nisqa_args["ms_n_fft"] = 512
nisqa_args["hop_length"] = 256
nisqa_args["ms_win_length"] = 512
nisqa_args["ckp"] = nisqa_args["ckp"][3:]

nisqa, h0_nisqa, c0_nisqa = model_init(nisqa_args)

/home/zakhar/miniconda3/envs/ems_dereverb/lib/python3.10/site-packages/torch/nn/modules/rnn.py:83: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=1 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


In [11]:
def pad_sequence(batch):
    if not batch:
        return torch.zeros(0), torch.zeros(0)

    input_signal, target_signal, noise, rir = zip(*batch)
        
    max_len_s = max(s.shape[-1] for s in input_signal)
    
    padded_input = torch.zeros(len(input_signal), max_len_s)
    padded_target = torch.zeros(len(target_signal), max_len_s)
    
    for i, s in enumerate(input_signal):
        padded_input[i, :s.shape[-1]] = s
        padded_target[i, :s.shape[-1]] = target_signal[i]

    return padded_input, padded_target


def collate_fn(batch):
    
    padded_input, padded_target = pad_sequence(batch)
        
    padded_input = padded_input.reshape(-1, padded_input.shape[-1])
    padded_target = padded_target.reshape(-1, padded_input.shape[-1])

    return padded_input, padded_target

In [12]:
test_dataloader = DataLoader(dataset, batch_size=1, shuffle=False, drop_last=False, collate_fn=collate_fn)

In [13]:
tmp_input = collate_fn((dataset[0],))

In [14]:
import time

def check_stream_inference(model, loader, window_size = SR, device="cpu"):
    model.eval()

    for _ in tqdm(range(10)):
        window = vorbis_window(N_FFTS).to(device)
        spec = torch.stft(
                tmp_input[0][..., :window_size],
                n_fft=N_FFTS,
                hop_length=HOP_LENGTH,
                # onesided=True,
                win_length=N_FFTS,
                window=window,
                return_complex=True,
                normalized=True,
                center=True
        ) 
        # tmp = model_eval_compile(model, spec, configs, device) # model(input_spec_, abs_spectrum, h0)
        output, _ = model_eval_old(model, spec, configs, device)

    result_nisqa_full = []
    result_rtf_full = []
    result_nisqa_chunk = []
    result_rtf_chunk = []
    with torch.no_grad():
        for signal, target, _, _ in tqdm(dataset):
            signal = signal.to(device)# .unsqueeze(0)
            target = target.to(device)# .unsqueeze(0)
            window = vorbis_window(N_FFTS).to(device)
    
            start_time = time.time()
            spec = torch.stft(
                signal,
                n_fft=N_FFTS,
                hop_length=HOP_LENGTH,
                # onesided=True,
                win_length=N_FFTS,
                window=window,
                return_complex=True,
                normalized=True,
                center=True
            )

            # spec = use_pcs(spec, N_FFTS)
            
            # for _ in range(10):
            # output, _ = model_eval_old(model, spec, configs, device)

            # # output = inv_pcs(output.abs(), output.angle())

            # window = vorbis_window(N_FFTS).to(device)
            # output = torch.istft(output, n_fft=N_FFTS, hop_length=HOP_LENGTH, win_length=N_FFTS,
            #                        window=window,
            #                        # onesided=True,
            #                        return_complex=False,
            #                        normalized=True,
            #                        center=True)
            
            end_time = time.time()
            
            result_rtf_full.append((signal.shape[-1] / SR) / (end_time - start_time))
            # nisqa_score, _, _ = process(output.detach().cpu(), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)
            # result_nisqa_full.append(nisqa_score)

            for j in range(0, signal.shape[-1], window_size):
                chunk = signal[..., j:j+window_size]
                
                if chunk.shape[-1] < window_size:
                    continue

                start_time = time.time()
                spec = torch.stft(
                    chunk,
                    n_fft=N_FFTS,
                    hop_length=HOP_LENGTH,
                    # onesided=True,
                    win_length=N_FFTS,
                    window=window,
                    return_complex=True,
                    normalized=True,
                    center=True
                )
                # print(spec.shape)

                output, _ = model_eval_old(model, spec, configs, device)

                window = vorbis_window(N_FFTS).to(device)
                output = torch.istft(output, n_fft=N_FFTS, hop_length=HOP_LENGTH, win_length=N_FFTS,
                                    window=window,
                                    # onesided=True,
                                    return_complex=False,
                                    normalized=True,
                                    center=True)
                end_time = time.time()


                result_rtf_chunk.append((chunk.shape[-1] / SR) / (end_time - start_time))
                # print(output.shape)
                # nisqa_score, _, _ = process(output.detach().cpu(), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)

                # result_nisqa_chunk.append(nisqa_score)
                

    # print(f"Mean nisqa for full audio: ", torch.stack(result_nisqa_full).mean(dim=0))
    # print(f"Mean rtf for full audio: ", torch.tensor(result_rtf_full).mean(dim=0), 1 / torch.tensor(result_rtf_full).mean(dim=0))
    # print("---" * 10)
    # print("Mean nisqa for \"stream\" audio: ", torch.stack(result_nisqa_chunk).mean(dim=0))
    print("Mean rtf for \"stream\" audio: ", torch.tensor(result_rtf_chunk).mean(dim=-1), 1 / torch.tensor(result_rtf_chunk).mean(dim=-1))

    return result_nisqa_full, result_rtf_full, result_nisqa_chunk, result_rtf_chunk

In [15]:
_, rtf_full, _, rtf_chunk = check_stream_inference(fspen, test_dataloader, window_size=N_FFTS * 5, device="cpu")

100%|██████████| 824/824 [05:09<00:00,  2.66it/s]

Mean rtf for "stream" audio:  tensor(6.957) tensor(0.144)


In [20]:
fspen.feature_merge_layer[0].weight.dtype

torch.float32

In [ ]:
rtf_full =  1 / torch.tensor(rtf_full).mean(dim=0)
rtf_chunk =  1 / torch.tensor(rtf_chunk).mean(dim=0)

In [ ]:
from torchmetrics.audio.pesq import PerceptualEvaluationSpeechQuality
from torch_stoi import NegSTOILoss

srmr = SpeechReverberationModulationEnergyRatio(fs=16_000, norm=False)
pesq = PerceptualEvaluationSpeechQuality(fs=16_000, mode="wb").to("cuda")
stoi = NegSTOILoss(16_000, use_vad=False, do_resample=False).to("cuda")
sisdr = ScaleInvariantSignalDistortionRatio().to("cuda")
dnsmos = DeepNoiseSuppressionMeanOpinionScore(16_000, False, device=DEVICE)

In [ ]:
OUTPUT_PATH = "data/fspen_48_overlap_no_ext_enhanced/"
GT_PATH = "data/gt_sim/"
INPUT_PATH = "data/input_sim/"

if not os.path.isdir(OUTPUT_PATH):
    os.mkdir(OUTPUT_PATH)

if not os.path.isdir(GT_PATH):
    os.mkdir(GT_PATH)

if not os.path.isdir(INPUT_PATH):
    os.mkdir(INPUT_PATH)

In [ ]:
from torchaudio.transforms import Resample
from thop import profile
from scipy.io.wavfile import write

def get_metrics(model, loader, device="cpu"):
    model.eval()
    
    model = model.to(device)
    
    nisqa_scores = []
    pesq_scores = []
    stoi_scores = []
    sisdr_scores = []
    srmr_scores = []
    dnsmos_scores = []
    macs_list = []
    with torch.no_grad():
        for ind, (signal, target) in tqdm(enumerate(loader)):
            signal = signal.to(device)
            target = target.to(device)
            window = vorbis_window(N_FFTS).to(device)

            if ind > 100:
                break
    
            spec = torch.stft(
                signal,
                n_fft=N_FFTS,
                hop_length=HOP_LENGTH,
                # onesided=True,
                win_length=N_FFTS,
                window=window,
                return_complex=True,
                normalized=True,
                center=True
            )

            # spec = use_pcs(spec, N_FFTS)
            # print(spec.device)
            output, _ = model_eval_old(model, spec, configs, device, hid_size=32)

            abs_s = spec.abs()
            input_s_ = torch.permute(torch.view_as_real(spec), dims=(0, 2, 3, 1))
            batch, frames, channels, frequency = input_s_.shape
            abs_s = torch.permute(abs_s, dims=(0, 2, 1))
            abs_s = torch.reshape(abs_s, shape=(batch, frames, 1, frequency))
            h0 = [[torch.zeros(configs.dual_path_extension["parameters"]["num_layers"], batch * configs.num_bands_out, configs.dual_path_extension["parameters"]["inter_hidden_size"]) for _ in range(8)] for _ in range(configs.dual_path_extension["num_modules"])]

            start = time.time()
            macs, _ = profile(model.cpu(), inputs=(input_s_.cpu(), abs_s.cpu(), h0), verbose=False)
            end = time.time()
            macs_list.append(macs / (end - start))

            model = model.to(device)

            # output = inv_pcs(output.abs(), output.angle())

            window = vorbis_window(N_FFTS).to(device)
            output = torch.istft(output, n_fft=N_FFTS, hop_length=HOP_LENGTH, win_length=N_FFTS,
                                   window=window,
                                   # onesided=True,
                                   return_complex=False,
                                   normalized=True,
                                   center=True)
            
            output = output / (output.abs().max() / signal.abs().max())
            
            min_l = min(output.shape[-1], target.shape[-1])
            nisqa_score, _, _ = process(output.detach().cpu(), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)

            output = output[:, :min_l]
            target = target[:, :min_l]

            write(OUTPUT_PATH + f"{ind}" + "_enhanced.wav", SR, output[0].cpu().detach().numpy())
            write(INPUT_PATH + f"{ind}" + "_noisy.wav", SR, signal[0].cpu().detach().numpy())
            write(GT_PATH + f"{ind}" + "_clean.wav", SR, target[0].cpu().detach().numpy())
            
            resampler = Resample(SR, 16_000)
            output = resampler(output.cpu()).cuda()
            target = resampler(target.cpu()).cuda()
            # min_l = min(output.shape[-1], target.shape[-1])

            stoi_score = stoi(output[..., :min_l], target[..., :min_l])
            srmr_score = srmr(output.detach().cpu())
            sisdr_score = sisdr(output, target)
            dnsmos_score = dnsmos(output.detach())

            try:
                pesq_score = pesq(output[..., :min_l], target[..., :min_l])
            except Exception as e:
                # print(min_l)
                # out_wave_ = output.reshape(-1)
                # target_ = target.reshape(-1)
                # write('exception_out.wav', SR, out_wave_.cpu().detach().numpy())
                # write('exception_in.wav', SR, target_.cpu().detach().numpy())
                continue

            nisqa_scores.append(nisqa_score[0])
            srmr_scores.append(srmr_score)
            stoi_scores.append(stoi_score.cpu())
            sisdr_scores.append(sisdr_score.cpu())
            pesq_scores.append(pesq_score.cpu())
            dnsmos_scores.append(dnsmos_score.cpu())

    print(len(nisqa_scores))
    result = {"nisqa": nisqa_scores, "stoi": stoi_scores, "sisdr": sisdr_scores, "srmr": srmr_scores, "pesq": pesq_scores, "dnsmos": dnsmos_scores}
        
    return result

In [ ]:
metrics = get_metrics(fspen, test_dataloader, device="cuda")

0it [00:00, ?it/s]

data/DS_10283_2791/clean_testset_wav/p257_365.wav data/rirs48_super_large_3_test/rir_77.wav 0.9228 no_file -100


1it [00:03,  3.34s/it]

data/DS_10283_2791/clean_testset_wav/p232_294.wav data/rirs48_super_large_3_test/rir_13.wav 0.8718 data/demand_test/TAU-urban-acoustic/TAU-urban-acoustic-scenes-2020-mobile-development.audio.14/TAU-urban-acoustic-scenes-2020-mobile-development/audio/street_traffic-milan-1087-41021-a.wav 10


2it [00:04,  2.33s/it]

...

data/DS_10283_2791/clean_testset_wav/p257_196.wav no_file 0.0 data/demand_test/TAU-urban-acoustic/TAU-urban-acoustic-scenes-2020-mobile-development.audio.15/TAU-urban-acoustic-scenes-2020-mobile-development/audio/tram-helsinki-276-8432-s3.wav 0


100it [02:18,  1.23s/it]

data/DS_10283_2791/clean_testset_wav/p232_299.wav data/rirs48_super_large_3_test/rir_77.wav 0.9228 data/demand_test/TAU-urban-acoustic/TAU-urban-acoustic-scenes-2020-mobile-development.audio.10/TAU-urban-acoustic-scenes-2020-mobile-development/audio/public_square-vienna-123-3644-a.wav 0


101it [02:19,  1.38s/it]

data/DS_10283_2791/clean_testset_wav/p257_233.wav data/rirs48_super_large_3_test/rir_30.wav 0.8592 data/demand_test/TAU-urban-acoustic/TAU-urban-acoustic-scenes-2020-mobile-development.audio.4/TAU-urban-acoustic-scenes-2020-mobile-development/audio/metro-barcelona-220-6644-a.wav 10
101


In [ ]:
print("NISQA:", torch.vstack(metrics["nisqa"]).mean(dim=0))
print("PESQ:", torch.vstack(metrics["pesq"]).mean(dim=0))
print("SRMR:", torch.vstack(metrics["srmr"]).mean(dim=0))
print("STOI:", -torch.vstack(metrics["stoi"]).mean(dim=0))
print("SI-SDR:", -torch.vstack(metrics["sisdr"]).mean(dim=0))
print("DNSMOS:", torch.vstack(metrics["dnsmos"]).mean(dim=0))
# print("MACs:", sum(metrics["macs"]) / len(metrics["macs"]))

NISQA: tensor([3.815, 4.095, 3.760, 3.854, 3.973])
PESQ: tensor([2.501])
SRMR: tensor([9.339])
STOI: tensor([0.892])
SI-SDR: tensor([12.855])
DNSMOS: tensor([3.171, 3.075, 3.865, 2.768], dtype=torch.float64)


NISQA: tensor([3.781, 4.034, 3.757, 3.831, 3.922])
PESQ: tensor([2.375])
SRMR: tensor([9.390])
STOI: tensor([0.887])
SI-SDR: tensor([13.084])
DNSMOS: tensor([3.183, 3.034, 3.857, 2.731], dtype=torch.float64)

In [ ]:
metrics = {k: [torch.vstack(v).mean(dim=0), ] for k, v in metrics.items()}
metrics["rtf_full"] = [rtf_full, ]
metrics["rtf_chunk"] = [rtf_chunk, ]

df = pd.DataFrame(metrics)
df.to_csv("fspen_48khz_overlap_no_ext_metrics_1986.csv", index=False)

NameError: name 'rtf_full' is not defined

In [ ]:
fspen

FullSubPathExtension(
  (full_band_encoder): FullBandEncoder(
    (full_band_encoder): ModuleList(
      (0): FullBandEncoderBlock(
        (conv): Conv1d(2, 4, kernel_size=(6,), stride=(2,), padding=(2,))
        (norm): BatchNorm1d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (activate): ELU(alpha=1.0)
      )
      (1): FullBandEncoderBlock(
        (conv): Conv1d(4, 16, kernel_size=(8,), stride=(2,), padding=(3,))
        (norm): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (activate): ELU(alpha=1.0)
      )
      (2): FullBandEncoderBlock(
        (conv): Conv1d(16, 32, kernel_size=(6,), stride=(2,), padding=(2,))
        (norm): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (activate): ELU(alpha=1.0)
      )
    )
    (global_features): Conv1d(32, 32, kernel_size=(1,), stride=(1,))
  )
  (sub_band_encoder): SubBandEncoder_baseline(
    (sub_band_encoders): ModuleList

In [ ]:
input_sig, gt, gt_noise, gt_rir = dataset[0]

In [ ]:
16_000 * (0.325 + 0.1625)

7800.000000000001

In [ ]:
from thop import profile
import time

window = vorbis_window(N_FFTS)

input_spec = torch.stft(
            input_sig[..., :16_000],
            n_fft=N_FFTS,
            hop_length=HOP_LENGTH,
            # onesided=True,
            win_length=N_FFTS,
            window=window,
            return_complex=True,
            normalized=True,
            center=True
        )

input_spec = input_spec.to("cpu")

abs_spectrum = input_spec.abs()
input_spec_ = torch.permute(torch.view_as_real(input_spec), dims=(0, 2, 3, 1))
batch, frames, channels, frequency = input_spec_.shape
abs_spectrum = torch.permute(abs_spectrum, dims=(0, 2, 1))
abs_spectrum = torch.reshape(abs_spectrum, shape=(batch, frames, 1, frequency))
h0 = [[torch.zeros(configs.dual_path_extension["parameters"]["num_layers"], batch * configs.num_bands_out, configs.dual_path_extension["parameters"]["inter_hidden_size"], device=input_spec.device) for _ in range(8)] for _ in range(configs.dual_path_extension["num_modules"])]

# output, hid_out = fspen(input_spec_, abs_spectrum, h0)
print(input_spec_.shape)

input_spec_ = torch.ones(1, 1, 2, 513)
abs_spectrum = torch.ones(1, 1, 1, 513)

fspen.eval()

start = time.time()
macs, params = profile(fspen.cpu(), inputs=(input_spec_, abs_spectrum, h0))
end = time.time()

torch.Size([1, 32, 2, 513])
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv1d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm1d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_gru() for <class 'torch.nn.modules.rnn.GRU'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.ConvTranspose1d'>.


In [ ]:
print("MACs: ", macs)
print("Params: ", params)

MACs:  5646528.0
Params:  95008.0


In [ ]:
print("MACs: ", macs / (end - start))
print("Params: ", params)

MACs:  94510730.667518
Params:  95008.0


In [ ]:
from src.utils import model_num_params

_, _ = model_num_params(fspen)

full_band_encoder.full_band_encoder.0.conv.weight ~  48        params ~ grad: True
full_band_encoder.full_band_encoder.0.conv.bias ~  4         params ~ grad: True
full_band_encoder.full_band_encoder.0.norm.weight ~  4         params ~ grad: True
full_band_encoder.full_band_encoder.0.norm.bias ~  4         params ~ grad: True
full_band_encoder.full_band_encoder.1.conv.weight ~  512       params ~ grad: True
full_band_encoder.full_band_encoder.1.conv.bias ~  16        params ~ grad: True
full_band_encoder.full_band_encoder.1.norm.weight ~  16        params ~ grad: True
full_band_encoder.full_band_encoder.1.norm.bias ~  16        params ~ grad: True
full_band_encoder.full_band_encoder.2.conv.weight ~  3.072     params ~ grad: True
full_band_encoder.full_band_encoder.2.conv.bias ~  32        params ~ grad: True
full_band_encoder.full_band_encoder.2.norm.weight ~  32        params ~ grad: True
full_band_encoder.full_band_encoder.2.norm.bias ~  32        params ~ grad: True
full_band_encode